# Polynomial system — Koopman estimation comparison

Computes prediction error vs number of training points L for all algorithms,
averaged over many random systems (different mu). Results are saved to a file
for separate plotting.

In [1]:
from pike import *
import os
import torch
import numpy as np
from tqdm.notebook import tqdm

In [2]:
n_vars = 3
degree = 5
seed = 19
torch.manual_seed(seed)
device = "cuda"

system = ClosedPoly(n_vars, degree, device)

# Koopman-invariant dictionary
pike = PIKE(system)
psi_defs_pike, K_pike = pike.generate()
n_obs_pike = len(psi_defs_pike)
P = n_vars  # number of parameters

# classic dictionary: all monomials up to max individual degree in the informed dictionary
d_max = max(max(max(psi.keys())) for psi in psi_defs_pike)
virt_sys = PolyParamAffineSystem(n_vars, d_max, [], device)
classic_psi_defs = [{exp: 1} for exp in virt_sys.exp_to_idx][1:]  # skip the constant observable
n_obs_classic = len(classic_psi_defs)

print(f"PIKE dictionary size: {n_obs_pike}")
print(f"Classic dictionary size (degree {d_max}): {n_obs_classic}")
print(f"Number of parameters P: {P}")

# Estimators
classic_ke = KoopmanEstimation(classic_psi_defs, n_vars, device)
pike_ke = KoopmanEstimation(psi_defs_pike, n_vars, device)

PIKE dictionary size: 8
Classic dictionary size (degree 4): 34
Number of parameters P: 3


## Pre-training for e-pEDMD and sparse-iEDMD

In [9]:
n_systems_train = 3 * n_vars
n_points_train = 1000

mus_train = torch.rand(n_systems_train, n_vars, device=device, dtype=torch.float64) * 4 - 5
X_train = torch.rand(n_systems_train, n_vars, n_points_train, device=device, dtype=torch.float64) * 10 - 5
X_dot_train = torch.zeros_like(X_train)
for i in range(n_systems_train):
    X_dot_train[i] = system(X_train[i], mus_train[i])

# e-pEDMD: learn K_i matrices from labeled multi-system data
K_est = pike_ke.fit_e_pEDMD(X_train, X_dot_train, mus_train)
print(f"e-pEDMD  K_est shape: {K_est.shape}")

# sparse-iEDMD: identify constant vs variable entries
K_c, K_mask = pike_ke.fit_sparse_iEDMD(X_train, X_dot_train)
pike_ke.precompute_sparse_structure(K_mask)
print(f"sparse-iEDMD  K_c shape: {K_c.shape},  K_mask shape: {K_mask.shape}")
print(f"  Free entries: {int(K_mask.sum())}/{K_mask.numel()}")

# Save K_c and K_mask
save_dir = "../results"
os.makedirs(save_dir, exist_ok=True)
np.savez(os.path.join(save_dir, "poly_sparse_matrices.npz"), K_c=K_c.cpu().numpy(), K_mask=K_mask.cpu().numpy())

e-pEDMD  K_est shape: torch.Size([4, 8, 8])
sparse-iEDMD  K_c shape: torch.Size([8, 8]),  K_mask shape: torch.Size([8, 8])
  Free entries: 10/64


## Main simulation loop

In [15]:
n_train = 500
n_test  = 300
n_sys   = 1000

results_file = os.path.join(save_dir, f"poly_errors_n{n_vars}_sys{n_sys}.npz")

if os.path.exists(results_file):
    print(f"Results already exist: {results_file}")
else:
    T_total = n_train + n_test

    # ── 1. Generate ALL systems at once ───────────────────────────────────
    X_all     = torch.rand(n_sys, n_vars, T_total, device=device, dtype=torch.float64) * 10 - 5
    mu_all    = torch.rand(n_sys, n_vars, device=device, dtype=torch.float64) * 4 - 5
    X_dot_all = torch.stack([system(X_all[i], mu_all[i]) for i in range(n_sys)])
    # (n_sys, n_vars, T_total)

    # ── 2. Lift ALL systems in one GPU pass ───────────────────────────────
    # Flatten (n_sys, n_vars, T) → (n_vars, n_sys*T), evaluate, reshape back to (n_sys, n_obs, T)
    X_flat  = X_all.permute(1, 0, 2).reshape(n_vars, n_sys * T_total)
    Xd_flat = X_dot_all.permute(1, 0, 2).reshape(n_vars, n_sys * T_total)

    psi_c_flat,  dpsi_c_flat  = classic_ke._resolve_lifted(X_flat, Xd_flat)
    psi_p_flat,  dpsi_p_flat  = pike_ke._resolve_lifted(X_flat, Xd_flat)
    del X_flat, Xd_flat

    psi_c  = psi_c_flat.reshape(n_obs_classic, n_sys, T_total).permute(1, 0, 2)
    dpsi_c = dpsi_c_flat.reshape(n_obs_classic, n_sys, T_total).permute(1, 0, 2)
    psi_p  = psi_p_flat.reshape(n_obs_pike,    n_sys, T_total).permute(1, 0, 2)
    dpsi_p = dpsi_p_flat.reshape(n_obs_pike,   n_sys, T_total).permute(1, 0, 2)
    del psi_c_flat, dpsi_c_flat, psi_p_flat, dpsi_p_flat

    psi_c_tr,  psi_c_te  = psi_c[:,  :, :n_train], psi_c[:,  :, n_train:]
    dpsi_c_tr, dpsi_c_te = dpsi_c[:, :, :n_train], dpsi_c[:, :, n_train:]
    psi_p_tr,  psi_p_te  = psi_p[:,  :, :n_train], psi_p[:,  :, n_train:]
    dpsi_p_tr, dpsi_p_te = dpsi_p[:, :, :n_train], dpsi_p[:, :, n_train:]

    # ── 3. Precompute structure ────────────────────────────────────────────
    K0_pike, Kp_pike = K_pike[0], K_pike[1:]   # (n_obs,n_obs), (P, n_obs,n_obs)
    K0_est,  Kp_est  = K_est[0],  K_est[1:]
    pike_ke.precompute_sparse_structure(K_mask)

    # ── 4. Accumulators ───────────────────────────────────────────────────
    errors_gEDMD        = torch.zeros(n_train, device=device, dtype=torch.float64)
    errors_iEDMD        = torch.zeros(n_train, device=device, dtype=torch.float64)
    errors_pEDMD        = torch.zeros(n_train, device=device, dtype=torch.float64)
    errors_e_pEDMD      = torch.zeros(n_train, device=device, dtype=torch.float64)
    errors_sparse_iEDMD = torch.zeros(n_train, device=device, dtype=torch.float64)
    condn_gEDMD   = torch.zeros(n_train, device=device, dtype=torch.float64)
    condn_iEDMD   = torch.zeros(n_train, device=device, dtype=torch.float64)
    condn_pEDMD   = torch.zeros(n_train, device=device, dtype=torch.float64)
    condn_e_pEDMD = torch.zeros(n_train, device=device, dtype=torch.float64)

    def pred_error(K_b, psi_te, dpsi_te):
        """Sum of per-system mean prediction errors. K_b: (n_sys, n_obs, n_obs)"""
        pred  = K_b @ psi_te                                               # (n_sys, n_obs, n_test)
        norms = torch.norm(pred[:, :n_vars] - dpsi_te[:, :n_vars], dim=1) # (n_sys, n_test)
        return norms.mean(dim=1).sum()                                     # scalar, sum over systems

    # ── 5. Single loop over l — all systems processed in parallel ─────────
    print(f"Running simulation (batched over {n_sys} systems)...")
    for l in tqdm(range(n_train)):
        L = l + 1

        pc  = psi_c_tr[:,  :, :L]   # (n_sys, n_obs_c, L)
        dpc = dpsi_c_tr[:, :, :L]
        pp  = psi_p_tr[:,  :, :L]   # (n_sys, n_obs_p, L)
        dpp = dpsi_p_tr[:, :, :L]

        # ── gEDMD ──────────────────────────────────────────────────────────
        Ac = pc @ pc.mT                                 # (n_sys, n_obs_c, n_obs_c)
        Kg = torch.linalg.solve(Ac, (dpc @ pc.mT).mT).mT
        errors_gEDMD[l] = pred_error(Kg, psi_c_te, dpsi_c_te)
        condn_gEDMD[l]  = torch.linalg.cond(Ac.cpu()).sum()

        # ── iEDMD ──────────────────────────────────────────────────────────
        Ap = pp @ pp.mT                                 # (n_sys, n_obs_p, n_obs_p)
        Ki = torch.linalg.solve(Ap, (dpp @ pp.mT).mT).mT
        errors_iEDMD[l] = pred_error(Ki, psi_p_te, dpsi_p_te)
        condn_iEDMD[l]  = torch.linalg.cond(Ap.cpu()).sum()

        # ── pEDMD ──────────────────────────────────────────────────────────
        # G[s,i,j] = trace(Kp[i]^T Kp[j] Mxx[s])  →  einsum
        Myx_p = (dpp - K0_pike @ pp) @ pp.mT           # (n_sys, n_obs_p, n_obs_p)
        Gp    = torch.einsum('iba,jbc,sca->sij', Kp_pike, Kp_pike, Ap)  # (n_sys, P, P)
        bp    = torch.einsum('iba,sba->si',       Kp_pike, Myx_p)       # (n_sys, P)
        mu_p  = torch.linalg.solve(Gp, bp.unsqueeze(-1)).squeeze(-1)    # (n_sys, P)
        Kp_b  = K0_pike + torch.einsum('si,iab->sab', mu_p, Kp_pike)   # (n_sys, n_obs_p, n_obs_p)
        errors_pEDMD[l] = pred_error(Kp_b, psi_p_te, dpsi_p_te)
        condn_pEDMD[l]  = torch.linalg.cond(Gp.cpu()).sum()

        # ── e-pEDMD ────────────────────────────────────────────────────────
        Myx_e = (dpp - K0_est @ pp) @ pp.mT
        Ge    = torch.einsum('iba,jbc,sca->sij', Kp_est, Kp_est, Ap)
        be    = torch.einsum('iba,sba->si',       Kp_est, Myx_e)
        mu_e  = torch.linalg.solve(Ge, be.unsqueeze(-1)).squeeze(-1)
        Ke_b  = K0_est + torch.einsum('si,iab->sab', mu_e, Kp_est)
        errors_e_pEDMD[l] = pred_error(Ke_b, psi_p_te, dpsi_p_te)
        condn_e_pEDMD[l]  = torch.linalg.cond(Ge.cpu()).sum()

        # ── sparse-iEDMD (still per-system, but CPU-based so fast) ─────────
        for s in range(n_sys):
            K_si = pike_ke.sparse_iEDMD(K_c, K_mask, psi=pp[s], dot_psi=dpp[s])
            pred = K_si @ psi_p_te[s]
            errors_sparse_iEDMD[l] += torch.mean(
                torch.norm(pred[:n_vars] - dpsi_p_te[s, :n_vars], dim=0)
            )

    # ── 6. Normalize and save ─────────────────────────────────────────────
    for arr in [errors_gEDMD, errors_iEDMD, errors_pEDMD, errors_e_pEDMD, errors_sparse_iEDMD,
                condn_gEDMD, condn_iEDMD, condn_pEDMD, condn_e_pEDMD]:
        arr /= n_sys

    os.makedirs("results", exist_ok=True)
    np.savez(results_file,
             errors_gEDMD=errors_gEDMD.cpu().numpy(),
             errors_iEDMD=errors_iEDMD.cpu().numpy(),
             errors_pEDMD=errors_pEDMD.cpu().numpy(),
             errors_e_pEDMD=errors_e_pEDMD.cpu().numpy(),
             errors_sparse_iEDMD=errors_sparse_iEDMD.cpu().numpy(),
             n_train=n_train, n_test=n_test,
             n_obs_pike=n_obs_pike, n_obs_classic=n_obs_classic,
             n_systems_train=n_systems_train, n_points_train=n_points_train,
             condn_gEDMD=condn_gEDMD.cpu().numpy(),
             condn_iEDMD=condn_iEDMD.cpu().numpy(),
             condn_pEDMD=condn_pEDMD.cpu().numpy(),
             condn_e_pEDMD=condn_e_pEDMD.cpu().numpy())
    print(f"Results saved to {results_file}")

Running simulation (batched over 1000 systems)...


  0%|          | 0/500 [00:00<?, ?it/s]

Results saved to ../results/poly_errors_n3_sys1000.npz


In [14]:
# Trajectory prediction
L = 200
mu = torch.rand(n_vars, device=device, dtype=torch.float64) * 4 - 5

# data
X_train = torch.rand(n_vars, L, device=device, dtype=torch.float64) * 10 - 5
X_dot_train = system(X_train, mu)

# psi
psi_train_classic, psi_dot_train_classic = classic_ke._resolve_lifted(X_train, X_dot_train)
psi_train_pike,    psi_dot_train_pike    = pike_ke._resolve_lifted(X_train, X_dot_train)

# gEDMD
K_gEDMD, _ = classic_ke.gEDMD(psi=psi_train_classic, dot_psi=psi_dot_train_classic)

# iEDMD
K_iEDMD, _ = pike_ke.gEDMD(psi=psi_train_pike, dot_psi=psi_dot_train_pike)

# pEDMD
K_pEDMD, _, _ = pike_ke.pEDMD(K_pike, psi=psi_train_pike, dot_psi=psi_dot_train_pike)

# e-pEDMD
K_e_pEDMD, _, _ = pike_ke.e_pEDMD(psi=psi_train_pike, dot_psi=psi_dot_train_pike, K_est=K_est)

# sparse-iEDMD
K_sparse_iEDMD = pike_ke.sparse_iEDMD(K_c, K_mask, psi=psi_train_pike, dot_psi=psi_dot_train_pike)

# Save K matrices
save_dir = "../results"
os.makedirs(save_dir, exist_ok=True)
np.savez(os.path.join(save_dir, f"poly_K_matrices_L{L}.npz"), K_gEDMD=K_gEDMD.cpu().numpy(),
 K_iEDMD=K_iEDMD.cpu().numpy(), K_pEDMD=K_pEDMD.cpu().numpy(), K_e_pEDMD=K_e_pEDMD.cpu().numpy(), K_sparse_iEDMD=K_sparse_iEDMD.cpu().numpy())


# Initial condition
x0 = torch.rand(n_vars, device=device, dtype=torch.float64) * 10 - 5
T_pred = 5
n_steps = 1000
t_eval = torch.linspace(0, T_pred, n_steps, device=device)
true_traj = system.simulate(x0, mu, t_eval)

def simulate_traj(K, psi_0, t_eval):
    At = t_eval.view(-1,1,1) * K.unsqueeze(0)  # (n_steps, n_obs, n_obs)
    psi_traj = torch.linalg.matrix_exp(At) @ psi_0  # (n_steps, n_obs)
    return psi_traj[:, :n_vars].squeeze().T  # (n_steps, n_vars)

psi_0_classic = classic_ke._eval_psi(x0.unsqueeze(-1))
psi_0_pike = pike_ke._eval_psi(x0.unsqueeze(-1))

# Simulate trajectories
pred_traj_gEDMD = simulate_traj(K_gEDMD, psi_0_classic, t_eval)
pred_traj_iEDMD = simulate_traj(K_iEDMD, psi_0_pike, t_eval)
pred_traj_pEDMD = simulate_traj(K_pEDMD, psi_0_pike, t_eval)
pred_traj_e_pEDMD = simulate_traj(K_e_pEDMD, psi_0_pike, t_eval)
pred_traj_sparse_iEDMD = simulate_traj(K_sparse_iEDMD, psi_0_pike, t_eval)

# Save trajectories
np.savez(os.path.join(save_dir, f"poly_trajectories_L{L}.npz"),
         true_traj=true_traj.cpu().numpy(),
         pred_traj_gEDMD=pred_traj_gEDMD.cpu().numpy(),
         pred_traj_iEDMD=pred_traj_iEDMD.cpu().numpy(),
         pred_traj_pEDMD=pred_traj_pEDMD.cpu().numpy(),
         pred_traj_e_pEDMD=pred_traj_e_pEDMD.cpu().numpy(),
         pred_traj_sparse_iEDMD=pred_traj_sparse_iEDMD.cpu().numpy(),
         t_eval=t_eval.cpu().numpy())